In [ ]:
print('===== NATIVE GELU FWD V2 - argtypes fixed =====')
print('===== NATIVE GELU FWD BITEXACT TEST =====')
print('===== NATIVE BIASGRAD BITEXACT V2 - 7 args =====')
print('===== NATIVE BIASGRAD BITEXACT TEST =====')
# 1/4 - clone main + prototype (BiasAddRows .tkv)
import subprocess, os
print('===== NATIVE BIASADD BITEXACT TEST =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so (has BiasAddRows) + .tkv-generated C .so
import subprocess, os
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/gelu_fwd_cpu.cpu.c'; so=ex+'/gelu_fwd_cpu.so'
# inject exact f32/f64 erf helper from tkv_bwd_ops.c into generated C
cc_path=ex+'/gelu_fwd_cpu.cpu.c'
txt=open(cc_path).read()
helper='''
static inline float tkv_gelu_erf_f(float v) {
    float arg = v / (float)1.4142135623730951;
    double sg = 1.0; if (arg < 0.0f) sg = -1.0;
    float ax = fabsf(arg);
    double t = 1.0 / (1.0 + 0.3275911 * (double)ax);
    double poly = 1.061405429;
    poly = poly * t + -1.453152027; poly = poly * t + 1.421413741;
    poly = poly * t + -0.284496736; poly = poly * t + 0.254829592;
    poly = poly * t;
    float e = (float)exp((double)((-ax) * ax));
    return (float)(sg * (1.0 - (double)e * poly));
}
static inline float tkv_gelu_f(float v) {
    float e = tkv_gelu_erf_f(v); float hf = 0.5f * v; float op = 1.0f + e;
    return hf * op;
}
'''
txt=txt.replace('#include <math.h>', '#include <math.h>\n'+helper)
open(cc_path,'w').write(txt)
print('helper injected =', 'tkv_gelu_f' in open(cc_path).read())
p2=subprocess.run('gcc -O2 -mavx2 -mfma -ffp-contract=off -fno-fast-math -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare GeluErfFwd
import ctypes, numpy as np
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so')
native.GeluErfFwd.argtypes=[ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.c_longlong,ctypes.c_longlong]
native.GeluErfFwd.restype=ctypes.c_int
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/gelu_fwd_cpu.so')
tkv.GeluErfFwd.argtypes=[ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.c_longlong,ctypes.c_longlong]
tkv.GeluErfFwd.restype=ctypes.c_int
rng=np.random.default_rng(7)
for lo,hi in [(0,64),(5,64),(0,1000),(3,777)]:
    x=rng.standard_normal(hi+4).astype(np.float32)
    a=np.zeros(hi-lo,np.float32); c=np.zeros(hi-lo,np.float32)
    native.GeluErfFwd(x.ctypes.data_as(ctypes.POINTER(ctypes.c_float)), x.size,
                      a.ctypes.data_as(ctypes.POINTER(ctypes.c_float)), a.size, lo, hi)
    tkv.GeluErfFwd(x.ctypes.data_as(ctypes.POINTER(ctypes.c_float)), x.size,
                   c.ctypes.data_as(ctypes.POINTER(ctypes.c_float)), c.size, lo, hi)
    d=np.count_nonzero(a.view(np.uint32)!=c.view(np.uint32))
    print('lo=%d hi=%d bitwise=%s diff=%d/%d maxabs=%.3g'%(lo,hi,d==0,d,a.size,np.abs(a-c).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> BiasAddRows can be replaced by .tkv-generated C.')